# 🛰️ rs-embed — Live Interactive Demo
### *Any Model · Any Place · Any Time* — SIGSPATIAL '26, UC Riverside

**rs-embed** turns any remote-sensing foundation model into a one-line
`ROI → embedding` service. This notebook is a hands-on console for four
interactive scenarios, told through **Riverside citrus** — the crop that put
this region on the map.

| | Scenario | What you do | rs-embed pillar |
|---|---|---|---|
| 🌍 | **Earth's Twins** | Click anywhere → find its look-alikes worldwide | *any place* |
| ⏳ | **Satellite Time Machine** | Slide through 2017→2024 → watch change | *any time* |
| 🎨 | **Instant Land-Cover** | Draw a box → unsupervised map; swap models | *any model* |
| 🏆 | **Model Showdown** | Rank models on citrus mapping, fairly | *fair comparison* |

> **Two ways to run.** `RUN_LIVE = False` (default) reads a pre-built cache and
> works with **no Earth Engine auth, no GPU, no network** — ideal for the booth.
> Set `RUN_LIVE = True` (with GEE auth + a GPU) to call the models live; any
> failure falls back to the cache automatically.


## Act 0 — the pain we're replacing

Using one of these foundation models *by hand* means: authenticating Google
Earth Engine, picking the right spectral bands **in the right order**, applying
the model's exact normalization, downloading checkpoints, and writing the
forward pass — **per model**. That's ≈ weeks of glue code, and the bugs are
*silent* (a wrong band order still returns a clean-looking vector).

Everything below is **one function call per model.**


In [ ]:
# ── Setup ───────────────────────────────────────────────────────────────────
import sys, json, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# Find the examples/ folder (works whether cwd is repo root or examples/).
HERE = Path.cwd()
for cand in [HERE, HERE / "examples", *HERE.parents]:
    if (cand / "iguide_demo_helpers.py").exists():
        sys.path.insert(0, str(cand)); break
import iguide_demo_helpers as H
from build_demo_cache import (
    HOTSPOTS, RIVERSIDE_GROVE, LANDCOVER_SCENES, NAMED_PLACES,
    MODEL_LIST, PRECOMPUTED_MODEL, TWIN_YEAR, YEARS, BUFFER_M,
)

# ⚙️  Flip to True with GEE auth + GPU to run models live.
RUN_LIVE = False

cache = H.DemoCache.find(HERE)
print("Cache dir :", cache.root)
print("Cache ready:", cache.available, "| RUN_LIVE:", RUN_LIVE)
if not cache.available:
    print("\n⚠️  No cache found. Build it once with:\n"
          "      python examples/build_demo_cache.py      # needs `earthengine authenticate`\n"
          "    Each scenario below will show a friendly notice until then.")

# Optional interactive widgets / map (graceful fallbacks if missing).
try:
    import ipywidgets as W
    HAS_W = True
except Exception:
    HAS_W = False
try:
    import geemap
    HAS_GEEMAP = True
except Exception:
    HAS_GEEMAP = False
print("ipywidgets:", HAS_W, "| geemap:", HAS_GEEMAP)

# Live API handle (loaded lazily, only when RUN_LIVE).
RS = None
if RUN_LIVE:
    try:
        import ee
        try:
            ee.Initialize()
        except Exception:
            ee.Authenticate(); ee.Initialize()
        from rs_embed import BBox, OutputSpec, PointBuffer, TemporalSpec, get_embedding
        RS = dict(get_embedding=get_embedding, PointBuffer=PointBuffer, BBox=BBox,
                  TemporalSpec=TemporalSpec, OutputSpec=OutputSpec)
        print("Live rs-embed ready ✔")
    except Exception as e:
        print("Live mode unavailable → using cache. Reason:", repr(e))
        RUN_LIVE = False

## 🌍 Scenario 1 — Earth's Twins  ·  *any place*

Click any point on Earth (or pick a named place). rs-embed turns it into one
vector with the precomputed **AlphaEarth** model, then finds the most
*embedding-similar* places on the planet. A Riverside grove's twins should be
the world's other citrus belts.


In [ ]:
bank = cache.twins_bank()
if bank is None:
    print("⚠️  twins_bank.npz missing — run: python build_demo_cache.py --only twins")
else:
    BV, BLON, BLAT = bank["vectors"], bank["lon"], bank["lat"]
    BNAME = bank["names"] if "names" in bank else np.array([f"pt{i}" for i in range(len(BLON))], object)
    BLL = np.stack([BLON, BLAT], axis=1)

    def _query_vector(lon, lat):
        if RUN_LIVE and RS is not None:
            try:
                emb = RS["get_embedding"](
                    PRECOMPUTED_MODEL,
                    spatial=RS["PointBuffer"](lon=lon, lat=lat, buffer_m=BUFFER_M),
                    temporal=RS["TemporalSpec"].year(TWIN_YEAR),
                    output=RS["OutputSpec"].pooled())
                return H.pooled_vector(emb.data), "live embedding"
            except Exception as e:
                print("live embed failed → snapping to cache:", repr(e))
        j = int(np.argmin(np.hypot(BLON - lon, BLAT - lat)))   # cached: snap click
        return BV[j], f"snapped to '{BNAME[j]}'"

    def find_twins(lon, lat, k=5):
        q, how = _query_vector(lon, lat)
        idx, sims = H.cosine_topk(q, BV, k=k, exclude_self_radius_deg=0.75,
                                  query_lonlat=(lon, lat), bank_lonlat=BLL)
        print(f"Query ({lon:.2f}, {lat:.2f})  [{how}]  → top {len(idx)} twins:")
        for r, (i, s) in enumerate(zip(idx, sims), 1):
            print(f"  {r}. {str(BNAME[i]):<32} ({BLON[i]:7.2f},{BLAT[i]:6.2f})  cos={s:.3f}")
        fig, ax = plt.subplots(figsize=(11, 5))
        ax.scatter(BLON, BLAT, s=6, c="#d9d9d9")
        ax.scatter(BLON[idx], BLAT[idx], s=70, c="#1f77b4", zorder=4, label="twins")
        ax.scatter([lon], [lat], s=170, marker="*", c="#d62728", zorder=5, label="query")
        for i in idx:
            ax.annotate(str(BNAME[i]), (BLON[i], BLAT[i]), fontsize=8,
                        xytext=(3, 3), textcoords="offset points")
        ax.set_xlim(-180, 180); ax.set_ylim(-60, 78)
        ax.set_title("🌍 Earth's Twins"); ax.legend(loc="lower left")
        plt.show()
        return idx, sims

    # Default render: twins of the Riverside grove.
    find_twins(*RIVERSIDE_GROVE)

    # Interactive: map click if geemap is present, else a place dropdown.
    if HAS_GEEMAP:
        m = geemap.Map(center=[33.97, -117.34], zoom=4)
        def _on_click(**kw):
            if kw.get("type") == "click":
                lat, lon = kw["coordinates"]; find_twins(lon, lat)
        m.on_interaction(_on_click)
        print("Click anywhere on the map ↓")
        display(m)
    elif HAS_W:
        dd = W.Dropdown(options=list(NAMED_PLACES), description="Place:")
        btn = W.Button(description="Find twins", button_style="info")
        out = W.Output()
        def _go(_):
            out.clear_output()
            with out: find_twins(*NAMED_PLACES[dd.value])
        btn.on_click(_go); display(W.HBox([dd, btn]), out)

## ⏳ Scenario 2 — Satellite Time Machine  ·  *any time*

Pick a change hotspot and watch its embedding evolve year by year. The
**change curve** (embedding distance from the first year) spikes exactly when
something happened on the ground — no labels, no change-detection model.
Local pick: the shrinking **Salton Sea**, ~1 hour from UCR.


In [ ]:
tm = cache.timemachine()
if tm is None:
    print("⚠️  timemachine.npz missing — run: python build_demo_cache.py --only timemachine")
else:
    hotspots = [str(s) for s in tm["hotspots"]] if "hotspots" in tm else \
               sorted({k[:-7] for k in tm if k.endswith("__grids")})

    def show_timemachine(slug):
        grids, years = tm[f"{slug}__grids"], tm[f"{slug}__years"]
        cc = H.change_curve(grids, baseline_index=0)
        n = len(years)
        fig, axes = plt.subplots(1, n + 1, figsize=(2.3 * (n + 1), 2.7))
        for j, yr in enumerate(years):
            axes[j].imshow(H.pca_rgb(grids[j])); axes[j].set_title(str(int(yr)), fontsize=9)
            axes[j].axis("off")
        axes[-1].plot(years, cc, marker="o", color="#d62728")
        axes[-1].set_title(f"change vs {int(years[0])}", fontsize=9)
        axes[-1].set_xlabel("year"); axes[-1].set_ylabel("1 − cos")
        fig.suptitle(f"⏳ Satellite Time Machine — {slug}", fontsize=12)
        plt.tight_layout(); plt.show()
        print(f"Biggest embedding change around {int(years[int(np.argmax(cc))])}.")

    show_timemachine(hotspots[0])
    if HAS_W and len(hotspots) > 1:
        dd = W.Dropdown(options=hotspots, description="Hotspot:")
        out = W.Output()
        def _go(change=None):
            out.clear_output()
            with out: show_timemachine(dd.value)
        dd.observe(lambda ch: _go() if ch["name"] == "value" else None)
        display(dd, out)

## 🎨 Scenario 3 — Instant Land-Cover  ·  *any model*

Embeddings are *features*. Cluster a region's embedding grid with KMeans and you
get an **unsupervised** land-cover map — citrus, urban, and desert separate out
with no labels. Swap the foundation model and watch the segmentation quality
change: same code, different model.


In [ ]:
scene = list(LANDCOVER_SCENES)[0]
lc = cache.landcover_scene(scene)
if lc is None:
    print(f"⚠️  landcover/{scene}.npz missing — run: python build_demo_cache.py --only landcover")
else:
    model_keys = [k[len("grid__"):] for k in lc if k.startswith("grid__")]

    def segment(model, k=6):
        grid = lc[f"grid__{model}"]
        _, rgb = H.kmeans_landcover(grid, k=k)
        pcs = H.pca_rgb(grid)
        fig, ax = plt.subplots(1, 2, figsize=(8, 4))
        ax[0].imshow(pcs); ax[0].set_title(f"{model}: PCA-RGB"); ax[0].axis("off")
        ax[1].imshow(rgb); ax[1].set_title(f"{model}: KMeans k={k} (no labels)"); ax[1].axis("off")
        plt.tight_layout(); plt.show()

    # Default: compare the first two models side by side.
    for mk in model_keys[:2]:
        segment(mk, 6)

    if HAS_W:
        dd = W.Dropdown(options=model_keys, description="Model:")
        ks = W.IntSlider(value=6, min=2, max=10, description="K:")
        out = W.Output()
        def _go(change=None):
            out.clear_output()
            with out: segment(dd.value, ks.value)
        dd.observe(lambda ch: _go() if ch["name"] == "value" else None)
        ks.observe(lambda ch: _go() if ch["name"] == "value" else None)
        display(W.HBox([dd, ks]), out)

## 🏆 Scenario 4 — Model Showdown  ·  *fair comparison*

The payoff. We map **citrus vs. non-citrus** across the Southern California
citrus belt (labels from USDA CDL) and ask: *which foundation model's
embeddings predict citrus best?* Crucially, `export_batch` fetched each point's
imagery **once and shared it** across all models (`dedup_reused`), and every
model is scored on the **same** train/test split — so the only thing that
differs is the model.


In [ ]:
sd = cache.showdown_dir()
labels = cache.showdown_labels()
bundle = sd / "citrus_export.npz"
if labels is None or not bundle.exists():
    print("⚠️  showdown bundle/labels missing — run: python build_demo_cache.py --only showdown")
else:
    feats, manifest = H.load_export_features(bundle)
    y = np.asarray(labels["y"]).ravel()
    board = H.classification_leaderboard(feats, y, test_size=0.3, seed=42)
    if not board:
        print("No models aligned with labels — re-run the showdown build.")
    else:
        print("Identical points, identical split — only the model differs.\n")
        print(f"{'rank':<5}{'model':<14}{'F1':>8}{'acc':>8}{'dim':>7}")
        for r, s in enumerate(board, 1):
            print(f"{r:<5}{s.model:<14}{s.f1:>8.3f}{s.accuracy:>8.3f}{s.dim:>7}")
        names = [s.model for s in board]; f1 = [s.f1 for s in board]
        fig, ax = plt.subplots(figsize=(7, 3.6))
        ax.barh(names[::-1], f1[::-1], color="#2e7d32")
        for i, v in enumerate(f1[::-1]):
            ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=8)
        ax.set_xlabel("macro-F1  (citrus vs other)")
        ax.set_title("🏆 Model Showdown — Riverside citrus mapping")
        ax.set_xlim(0, 1.05); plt.tight_layout(); plt.show()
        reused = [m.get("inputs", {}).get("dedup_reused")
                  for m in manifest.get("models", []) if isinstance(m.get("inputs"), dict)]
        print(f"\nFairness check — imagery fetched once & shared across models "
              f"(dedup_reused observed: {bool(any(reused))}).")

## Takeaways

- **One interface, many models.** Each scenario swapped models by changing a
  single string — no per-model glue, no silent band-order bugs.
- **Any place, any time.** Twins search and the time machine are just
  `(lon, lat, year) → vector`.
- **Reproducible science.** The Showdown ran on a shared, audited
  `export_batch` manifest — the fair comparison is the *default*, not extra work.

**Run it live:** set `RUN_LIVE = True` above (after `earthengine authenticate`)
to compute any cell against live imagery.
&nbsp; · &nbsp; Landing page: `examples/iguide_demo.html`
&nbsp; · &nbsp; Paper & docs: <https://cybergis.github.io/rs-embed/>
